<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта: Система способов оплаты</h2>

----

### Вариант задания № 16


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Разработка системы различных способов оплаты на базе класса PaymentMethod. Система предусматривает онлайн-оплату, банковский перевод и оплату наличными.

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) и реализуйте простое, сложное и множественное наследование

<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [1]:

// 1. Создание объектов (простое наследование)
BankTransfer bank = new BankTransfer(1, "Банковский перевод", 500, 1.5m, "ПАО Сбербанк, Р/С 407...");
CashPayment cash = new CashPayment(2, "Наличные", 50, 0m, "Касса №1 (ул. Мира, 10)");

// Создаем объект сложного наследования (Карта -> Онлайн -> Базовый метод)
CardPayment card = new CardPayment(3, "Банковская карта онлайн", 100, 2.0m, "https://securepay.ru", "2200-****-****-1234");

// 2. Вывод деталей способов оплаты
Console.WriteLine("--- Информация о способах оплаты ---");
bank.GetPaymentDetails();
Console.WriteLine($"Реквизиты: {bank.BankData}\n");

cash.GetPaymentDetails();
Console.WriteLine($"Точка оплаты: {cash.CashPickupPoint}\n");

card.GetPaymentDetails();
Console.WriteLine($"URL: {card.PaymentUrl}, Карта: {card.CardNumber}\n");

// 3. Проверка множественного наследования (методы интерфейсов)
Console.WriteLine("--- Проверка интерфейсов (множественное наследование) ---");
card.SendReceipt("student@mail.ru");
card.Refund(150);

// 4. Взаимодействие через Заказ
Console.WriteLine("\n--- Взаимодействие объектов с классом Order ---");
Order order1 = new Order(101, 1000);
Order order2 = new Order(102, 30); // сумма меньше минимальной (проверка условия)

order1.Pay(card);
order2.Pay(card);


// =========================================================================
// ИНТЕРФЕЙСЫ (ДЛЯ МНОЖЕСТВЕННОГО НАСЛЕДОВАНИЯ)
// =========================================================================

public interface IReceiptable
{
    void SendReceipt(string email);
}

public interface IRefundable
{
    void Refund(decimal amount);
}


// =========================================================================
// БАЗОВЫЙ КЛАСС
// =========================================================================

public class PaymentMethod
{
    private int _paymentMethodId;
    private string _methodName;
    private decimal _minAmount;
    private decimal _commission; // Добавлен атрибут 1: комиссия в %

    public int PaymentMethodId
    {
        get { return _paymentMethodId; }
        set { _paymentMethodId = value; }
    }

    public string MethodName
    {
        get { return _methodName; }
        set { _methodName = value; }
    }

    public decimal MinAmount
    {
        get { return _minAmount; }
        set { _minAmount = value; }
    }

    public decimal Commission
    {
        get { return _commission; }
        set { _commission = value; }
    }

    // Конструктор базового класса
    public PaymentMethod(int paymentMethodId, string methodName, decimal minAmount, decimal commission)
    {
        PaymentMethodId = paymentMethodId;
        MethodName = methodName;
        MinAmount = minAmount;
        Commission = commission;
    }

    public bool CheckMinimumAmount(decimal amount)
    {
        return amount >= MinAmount;
    }

    // Добавлен метод 1: расчет комиссии
    public decimal CalculateCommission(decimal amount)
    {
        return amount * (Commission / 100);
    }

    public virtual void ProcessPayment(decimal amount)
    {
        if (CheckMinimumAmount(amount))
        {
            decimal comm = CalculateCommission(amount);
            Console.WriteLine($"Успешно: платеж на сумму {amount} руб. (комиссия: {comm} руб.) проведен через '{MethodName}'.");
        }
        else
        {
            Console.WriteLine($"Ошибка: сумма {amount} руб. меньше минимальной ({MinAmount} руб.) для '{MethodName}'.");
        }
    }

    public virtual void GetPaymentDetails()
    {
        Console.WriteLine($"ID способа: {PaymentMethodId} | Название: {MethodName} | Мин. сумма: {MinAmount} руб. | Комиссия: {Commission}%");
    }
}


// =========================================================================
// ПРОСТОЕ НАСЛЕДОВАНИЕ (ПРОИЗВОДНЫЕ КЛАССЫ)
// =========================================================================

// 1. Онлайн-оплата (+ множественное наследование через IReceiptable)
public class OnlinePayment : PaymentMethod, IReceiptable
{
    private string _paymentUrl;

    public string PaymentUrl
    {
        get { return _paymentUrl; }
        set { _paymentUrl = value; }
    }

    public OnlinePayment(int paymentMethodId, string methodName, decimal minAmount, decimal commission, string paymentUrl)
        : base(paymentMethodId, methodName, minAmount, commission)
    {
        PaymentUrl = paymentUrl;
    }

    // Метод интерфейса IReceiptable
    public void SendReceipt(string email)
    {
        Console.WriteLine($"Электронный чек отправлен на почту: {email}");
    }
}

// 2. Банковский перевод
public class BankTransfer : PaymentMethod
{
    private string _bankData;

    public string BankData
    {
        get { return _bankData; }
        set { _bankData = value; }
    }

    public BankTransfer(int paymentMethodId, string methodName, decimal minAmount, decimal commission, string bankData)
        : base(paymentMethodId, methodName, minAmount, commission)
    {
        BankData = bankData;
    }
}

// 3. Наличные
public class CashPayment : PaymentMethod
{
    private string _cashPickupPoint;

    public string CashPickupPoint
    {
        get { return _cashPickupPoint; }
        set { _cashPickupPoint = value; }
    }

    public CashPayment(int paymentMethodId, string methodName, decimal minAmount, decimal commission, string cashPickupPoint)
        : base(paymentMethodId, methodName, minAmount, commission)
    {
        CashPickupPoint = cashPickupPoint;
    }
}


// =========================================================================
// СЛОЖНОЕ НАСЛЕДОВАНИЕ (CardPayment -> OnlinePayment -> PaymentMethod)
// + МНОЖЕСТВЕННОЕ НАСЛЕДОВАНИЕ (реализует еще и IRefundable)
// =========================================================================

public class CardPayment : OnlinePayment, IRefundable
{
    private string _cardNumber; // Добавлен атрибут 2: номер карты

    public string CardNumber
    {
        get { return _cardNumber; }
        set { _cardNumber = value; }
    }

    public CardPayment(int paymentMethodId, string methodName, decimal minAmount, decimal commission, string paymentUrl, string cardNumber)
        : base(paymentMethodId, methodName, minAmount, commission, paymentUrl)
    {
        CardNumber = cardNumber;
    }

    // Метод интерфейса IRefundable
    public void Refund(decimal amount)
    {
        Console.WriteLine($"Оформлен возврат {amount} руб. на карту {CardNumber}");
    }
}


// =========================================================================
// КЛАСС ЗАКАЗА (ДЛЯ ВЗАИМОДЕЙСТВИЯ ОБЪЕКТОВ)
// =========================================================================

public class Order
{
    public int OrderId { get; set; }
    public decimal TotalAmount { get; set; }

    public Order(int orderId, decimal totalAmount)
    {
        OrderId = orderId;
        TotalAmount = totalAmount;
    }

    public void Pay(PaymentMethod paymentMethod)
    {
        Console.WriteLine($"\nПопытка оплаты заказа №{OrderId} на сумму {TotalAmount} руб.");
        paymentMethod.ProcessPayment(TotalAmount);
    }
}

--- Информация о способах оплаты ---
ID способа: 1 | Название: Банковский перевод | Мин. сумма: 500 руб. | Комиссия: 1.5%
Реквизиты: ПАО Сбербанк, Р/С 407...

ID способа: 2 | Название: Наличные | Мин. сумма: 50 руб. | Комиссия: 0%
Точка оплаты: Касса №1 (ул. Мира, 10)

ID способа: 3 | Название: Банковская карта онлайн | Мин. сумма: 100 руб. | Комиссия: 2.0%
URL: https://securepay.ru, Карта: 2200-****-****-1234

--- Проверка интерфейсов (множественное наследование) ---
Электронный чек отправлен на почту: student@mail.ru
Оформлен возврат 150 руб. на карту 2200-****-****-1234

--- Взаимодействие объектов с классом Order ---

Попытка оплаты заказа №101 на сумму 1000 руб.
Успешно: платеж на сумму 1000 руб. (комиссия: 20.00 руб.) проведен через 'Банковская карта онлайн'.

Попытка оплаты заказа №102 на сумму 30 руб.
Ошибка: сумма 30 руб. меньше минимальной (100 руб.) для 'Банковская карта онлайн'.
